## Data Ingestion Lab

In [1]:
!uv pip install chroma langchain langchain-openai langchain-text-splitters langchain-chroma langchain-community pyyaml

Using Python 3.12.12 environment at: /Users/davidinyang-etoh/Projects/ai-projects/ai-playground/.venv
Audited 7 packages in 62ms


### Import modules and packages

In [2]:
import os
import re
import yaml
import glob
from pathlib import Path
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import MarkdownTextSplitter

from dotenv import load_dotenv

load_dotenv(override=True)

True

In [3]:
MODEL = "gpt-4.1-nano"

EMBEDDING_MODEL = "text-embedding-3-small"

# Sized to keep most sections whole (leader bios ~900 chars, core values ~1100 chars).
# Large sections (news items, leadership page) still split, but with enough overlap
# to preserve context across the boundary.
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 150

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

In [4]:
DOMAIN = "veroliq.com"

DB_NAME = str(Path("db/vector_db"))
KNOWLEDGE_BASE_PATH = str(f"data/{DOMAIN}")

print(f"Loading documents from {KNOWLEDGE_BASE_PATH}")

Loading documents from data/veroliq.com


### Pre-processing: frontmatter parsing + boilerplate cleaning

In [5]:
from datetime import date

def parse_frontmatter(text: str) -> tuple[dict, str]:
    """Strip YAML frontmatter from markdown. Returns (metadata_dict, body)."""
    if not text.startswith("---\n"):
        return {}, text
    end = text.find("\n---", 4)
    if end == -1:
        return {}, text
    try:
        metadata = yaml.safe_load(text[4:end]) or {}
    except yaml.YAMLError:
        metadata = {}
    return metadata, text[end + 4:].lstrip("\n")



def _chroma_friendly_scalar(v):
    """Chroma metadata must be str/int/float/bool/list/None — YAML timestamps become datetime."""
    if v is None:
        return ""
    if isinstance(v, date):
        return v.isoformat()
    return v


def preprocess_documents(documents):
    """
    For each loaded document:
      1. Parse YAML frontmatter → LangChain metadata fields
      2. Replace page_content with cleaned body (no frontmatter, no boilerplate)
    """
    for doc in documents:
        fm, body = parse_frontmatter(doc.page_content)
        if fm:
            doc.metadata.update({
                "url":           fm.get("url", ""),
                "page_title":    fm.get("title", ""),
                "breadcrumb":    fm.get("breadcrumb", ""),
                # Chroma requires scalar values — join list as comma string
                "path_segments": ",".join(fm.get("path_segments") or []),
                "crawled_at":    _chroma_friendly_scalar(fm.get("crawled_at", "")),
            })
        doc.page_content = body or doc.page_content
    return documents

In [6]:
def fetch_documents():
    folders = glob.glob(str(Path(KNOWLEDGE_BASE_PATH)))
    documents = []
    for folder in folders:
        doc_type = os.path.basename(folder)
        loader = DirectoryLoader(
            folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"}
        )
        folder_docs = loader.load()
        for doc in folder_docs:
            doc.metadata["doc_type"] = doc_type
            documents.append(doc)
    return documents

documents = fetch_documents()

print(f"Found {len(documents)} documents in {KNOWLEDGE_BASE_PATH}")

Found 1 documents in data/veroliq.com


In [7]:
def fix_dangling_headings(chunks: list) -> list:
    """
    Two-pass fix for heading/content misalignment after MarkdownTextSplitter:

    Case 1 — heading-only chunk: accumulate into a pending prefix and skip
              the chunk; prefix is prepended to the next content chunk.
    Case 2 — chunk ends with heading(s) after body content: strip the trailing
              headings (splitter overlap already carries them into the next
              chunk's start, so no manual forwarding needed).
    """
    result = []
    pending_prefix = ""

    for chunk in chunks:
        text = chunk.page_content
        non_blank = [l.strip() for l in text.splitlines() if l.strip()]

        # Case 1: entirely heading lines — hold as prefix, don't emit
        if non_blank and all(l.startswith("#") for l in non_blank):
            pending_prefix += text.strip() + "\n\n"
            continue

        # Prepend any accumulated headings from prior heading-only chunk(s)
        if pending_prefix:
            chunk.page_content = pending_prefix + text
            text = chunk.page_content
            pending_prefix = ""

        # Case 2: strip trailing headings that follow body content
        lines = text.split("\n")
        i = len(lines) - 1
        while i >= 0 and (not lines[i].strip() or lines[i].strip().startswith("#")):
            i -= 1
        # i is now the last line with real body content
        if i < len(lines) - 1:
            has_trailing = any(
                l.strip().startswith("#") for l in lines[i + 1:] if l.strip()
            )
            if has_trailing:
                chunk.page_content = "\n".join(lines[: i + 1]).rstrip()

        if chunk.page_content.strip():
            result.append(chunk)

    return result


def create_chunks(documents):
    text_splitter = MarkdownTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    chunks = text_splitter.split_documents(documents)
    chunks = fix_dangling_headings(chunks)

    enriched = []
    for chunk in chunks:
        # Drop near-empty chunks
        if len(chunk.page_content.strip()) < 60:
            continue

        # Prepend breadcrumb as context prefix for the embedding.
        # Falls back to page_title, then doc_type.
        breadcrumb = (
            chunk.metadata.get("breadcrumb")
            or chunk.metadata.get("page_title")
            or chunk.metadata.get("doc_type", "")
        )
        if breadcrumb:
            chunk.page_content = f"[{breadcrumb}]\n\n{chunk.page_content}"

        enriched.append(chunk)

    return enriched

In [8]:
def create_embeddings(chunks):
    if os.path.exists(DB_NAME):
        Chroma(persist_directory=DB_NAME,
               embedding_function=embeddings).delete_collection()

    vectorstore = Chroma.from_documents(
        documents=chunks, embedding=embeddings, persist_directory=DB_NAME
    )

    collection = vectorstore._collection
    count = collection.count()

    sample_embedding = collection.get(limit=1, include=["embeddings"])[
        "embeddings"][0]
    dimensions = len(sample_embedding)
    print(
        f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")
    return vectorstore

In [9]:
documents = fetch_documents()
documents = preprocess_documents(documents)
chunks = create_chunks(documents)
create_embeddings(chunks)
print("Ingestion complete")

There are 6 vectors with 1,536 dimensions in the vector store
Ingestion complete


In [20]:
def get_chunks(query):
    vectorstore = Chroma(
        embedding_function=embeddings,
        persist_directory=DB_NAME
    )
    return vectorstore.similarity_search(query)


results = get_chunks("contact veroliq?")

print(results)

[Document(id='e5b38ba4-d9d7-4596-9cb6-bb00c5d02156', metadata={'crawled_at': '2026-03-30T09:03:33.754845', 'breadcrumb': 'www.veroliq.com', 'source': 'data/veroliq.com/index.md', 'page_title': 'Veroliq', 'path_segments': '', 'doc_type': 'veroliq.com', 'url': 'https://www.veroliq.com/'}, page_content='[www.veroliq.com]\n\n# Veroliq\n\n$ /$\n\n## Your visitors ask questions.Most leave without an answer.\n\nVeroliq reads your website and trains Vera AI to answer questions and capture lead emails — 24/7, the moment a visitor shows buying intent. sarah.k@techstartup.io LEADS THIS MONTH 41 What are your pricing plans? We have 3 plans from Free to $15/mo — I can walk you through which fits best. What\'s your email so I can send the details? Works on every platform 0 % of visitors leave without converting The silent majority 0 min average setup time No developer needed 0 % average lift in leads captured Across all customers 0 /mo Starter plan Less than a coffee Built for\n\n## The people who c

In [21]:
from IPython.display import Markdown
chunk = results[0]

print(len(results))

chunk_count = 1;
for result in results:
    print(f"No: {chunk_count}")
    print("="*100)
    print(result.metadata)
    print("-"*100)
    display(Markdown(result.page_content))
    chunk_count += 1

4
No: 1
{'crawled_at': '2026-03-30T09:03:33.754845', 'breadcrumb': 'www.veroliq.com', 'source': 'data/veroliq.com/index.md', 'page_title': 'Veroliq', 'path_segments': '', 'doc_type': 'veroliq.com', 'url': 'https://www.veroliq.com/'}
----------------------------------------------------------------------------------------------------


[www.veroliq.com]

# Veroliq

$ /$

## Your visitors ask questions.Most leave without an answer.

Veroliq reads your website and trains Vera AI to answer questions and capture lead emails — 24/7, the moment a visitor shows buying intent. sarah.k@techstartup.io LEADS THIS MONTH 41 What are your pricing plans? We have 3 plans from Free to $15/mo — I can walk you through which fits best. What's your email so I can send the details? Works on every platform 0 % of visitors leave without converting The silent majority 0 min average setup time No developer needed 0 % average lift in leads captured Across all customers 0 /mo Starter plan Less than a coffee Built for

## The people who can't afford to miss a lead

🚀 Startup founders

### Sell while you build

You're coding, fundraising, and wearing every hat. Vera handles every pricing question, demo request, and "how does this work?" — capturing lead emails before the visitor bounces. 🛍️ Small businesses

### No live chat team? No problem.

Visitors browse at 11pm when your team is offline. Vera answers product questions, shipping queries, and booking requests — and sends you a warm lead list every morning. 🏢 Digital agencies

No: 2
{'breadcrumb': 'www.veroliq.com', 'crawled_at': '2026-03-30T09:03:33.754845', 'path_segments': '', 'doc_type': 'veroliq.com', 'source': 'data/veroliq.com/index.md', 'page_title': 'Veroliq', 'url': 'https://www.veroliq.com/'}
----------------------------------------------------------------------------------------------------


[www.veroliq.com]

## Questions aboutVeroliq

How long does setup actually take? What if Vera gives a wrong or incomplete answer? Does it work with WordPress, Webflow, or Shopify? What is an AI Action? Is visitor data safe and GDPR-compliant? Can I use my own OpenAI or Google AI key?

## Stop losing leads to the silence.

Every visitor who leaves without an answer is a lead you didn't capture. Veroliq fixes that — in one afternoon, for $5 a month. Free plan available · No credit card · Live in 5 minutes $ /$

No: 3
{'crawled_at': '2026-03-30T09:03:33.754845', 'breadcrumb': 'www.veroliq.com', 'url': 'https://www.veroliq.com/', 'source': 'data/veroliq.com/index.md', 'page_title': 'Veroliq', 'path_segments': '', 'doc_type': 'veroliq.com'}
----------------------------------------------------------------------------------------------------


[www.veroliq.com]

## Not another live chat. Not a chatbot builder.

Purpose-built to answer questions and capture leads — without the complexity or the $100/month price tag. 5-min setup, no developer | ✓ | ✕ | Partial | Learns from your website automatically | ✓ | ✕ | ✕ | Captures lead emails in conversation | ✓ | ✓ | Partial | Zero ongoing training or FAQ writing | ✓ | ✕ | ✕ | Multi-site management | ✓ | ✓ | ✕ | AI answer quality analytics | ✓ | ✕ | ✕ | Starts at $5/month | ✓ | ✕ | ✓ Pricing

## Start free. Pay only when you grow.

No contracts. No setup fees. Cancel any time. Free For solo founders testing the waters 1 website 350 chats / month Unlimited leads Basic analytics Veroliq branding Most popular Starter For founders ready to capture every lead 3 websites 5,000 chats / month Unlimited leads 3 AI actions Remove Veroliq branding Email notifications Growth For teams with multiple sites 10 websites 20,000 chats / month Unlimited leads 5 AI actions Advanced analytics Escalation alerts Priority support All plans include SSL, 99.9% uptime SLA, and GDPR-compliant data handling FAQ

No: 4
{'breadcrumb': 'www.veroliq.com', 'page_title': 'Veroliq', 'source': 'data/veroliq.com/index.md', 'crawled_at': '2026-03-30T09:03:33.754845', 'doc_type': 'veroliq.com', 'url': 'https://www.veroliq.com/', 'path_segments': ''}
----------------------------------------------------------------------------------------------------


[www.veroliq.com]

### AI on every client site

Manage all client websites from one dashboard. Per-site analytics, individual knowledge bases, and custom widget styling. Add a new revenue line to your agency. What Veroliq + Vera do

## One script tag. A lead machine that never sleeps.

No FAQ writing. No chatbot logic to design. Veroliq reads everything on your website and builds the knowledge base Vera uses automatically.

### Reads your website. Answers questions instantly.

Paste one script tag. Veroliq crawls your pricing pages, FAQs, docs, and product pages — then Vera answers visitor questions with that exact content. No setup beyond the script tag.

### Captures lead emails in conversation

When Vera detects buying intent, the email is captured naturally inside the chat — no pop-up, no form. Lead lands in your dashboard instantly. S sarah@techcorp.io J james@startup.com P priya@venture.co

### Answers at 3am on a Sunday

Time zones and bank holidays don't stop Vera. Visitors get a real answer immediately — and you wake up to warm leads.

### Tells you what's working

Which pages drive the most chats, which questions the AI can't answer, and exactly how many leads you captured this week.